# Equality Constrained Optimization

Solving constrained optimization using the Lagrangian method with GMM.

**Problem:** Gasoline blending to minimize cost while meeting octane and mass balance constraints.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import warnings

# Configure JAX to use CPU (avoids GPU/XLA compatibility issues)
import os
os.environ['JAX_PLATFORMS'] = 'cpu'

# Import reusable utilities from local module
from generative_optimization import (
    generate_samples,
    estimate_sample_size,
    best_gmm,
    best_gmm_bic,
    cluster_stats,
    validate_gmm
)

# Figure settings
mpl.rcParams['figure.facecolor'] = 'white'
mpl.rcParams['axes.facecolor'] = 'white'
mpl.rcParams['figure.dpi'] = 150

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore', category=RuntimeWarning)

---
<a id='constrained'></a>
## 4. Constrained Optimization

<a id='equality'></a>
### 4.1 Equality Constraints: Gasoline Blending

For equality constraints, we use a Lagrangian approach and condition on all Lagrangian derivatives being zero.

**Problem:** A refinery blends three gasoline components to minimize cost while meeting an octane specification:

$$\min_{x_1, x_2, x_3} \quad C = c_1 x_1 + c_2 x_2 + c_3 x_3$$

Subject to:
- Octane constraint: $\text{RON}_1 x_1 + \text{RON}_2 x_2 + \text{RON}_3 x_3 = 87$
- Mass balance: $x_1 + x_2 + x_3 = 1$

| Component | Cost ($/gal) | RON |
|-----------|-------------|-----|
| Reformate ($x_1$) | 2.50 | 95 |
| Isomerate ($x_2$) | 2.00 | 82 |
| Alkylate ($x_3$)  | 3.00 | 94 |

In [ ]:
# First, solve with scipy for comparison
from scipy.optimize import minimize

# Component parameters
costs = np.array([2.50, 2.00, 3.00])  # $/gal
rons = np.array([95, 82, 94])  # Research Octane Number
ron_target = 87  # Regular gasoline specification

def objective(x):
    """Blending cost"""
    return np.dot(costs, x)

def octane_constraint(x):
    """RON constraint: sum(RON_i * x_i) = RON_target"""
    return np.dot(rons, x) - ron_target

def mass_balance(x):
    """Mass balance: sum(x_i) = 1"""
    return np.sum(x) - 1

constraints = [
    {'type': 'eq', 'fun': octane_constraint},
    {'type': 'eq', 'fun': mass_balance}
]

# Bounds: 0 <= x_i <= 1
bounds = [(0, 1), (0, 1), (0, 1)]

sol = minimize(objective, [0.33, 0.33, 0.34], 
               constraints=constraints, bounds=bounds, method='SLSQP')

print("Scipy solution (SLSQP):")
print(f"  Reformate (x1) = {sol.x[0]:.4f}")
print(f"  Isomerate (x2) = {sol.x[1]:.4f}")
print(f"  Alkylate  (x3) = {sol.x[2]:.4f}")
print(f"  Total cost = ${sol.fun:.4f}/gal")
print(f"  RON achieved = {np.dot(rons, sol.x):.2f}")

In [ ]:
# Define Lagrangian and use JAX for derivatives
from jax import jacobian, vmap
import jax.numpy as jnp

# Convert to JAX arrays for autodiff
costs_jax = jnp.array([2.50, 2.00, 3.00])
rons_jax = jnp.array([95.0, 82.0, 94.0])
ron_target_jax = 87.0

def lagrangian(Y):
    """
    Lagrangian: L = cost + lambda1 * g1 + lambda2 * g2
    
    Y = [x1, x2, x3, lambda1, lambda2]
    g1 = octane constraint (RON - target)
    g2 = mass balance (sum - 1)
    """
    x = Y[0:3]
    lambda1, lambda2 = Y[3], Y[4]
    
    # Objective: blending cost
    cost = jnp.dot(costs_jax, x)
    
    # Constraints
    g1 = jnp.dot(rons_jax, x) - ron_target_jax  # Octane constraint
    g2 = jnp.sum(x) - 1.0  # Mass balance
    
    return cost + lambda1 * g1 + lambda2 * g2

print("Lagrangian defined with 2 equality constraints")
print("Variables: [x1, x2, x3, λ1, λ2]")
print("KKT conditions: ∂L/∂x1 = ∂L/∂x2 = ∂L/∂x3 = ∂L/∂λ1 = ∂L/∂λ2 = 0")

In [ ]:
# Generate samples of [x1, x2, x3, lambda1, lambda2] and compute Lagrangian derivatives
# Bounds for blend fractions [0, 1] and Lagrange multipliers (estimate scale)
bounds = [[0, 1], [0, 1], [0, 1], [-0.1, 0.1], [-5, 5]]
Y = generate_samples(bounds, n_samples=256, seed=42)

# Compute Jacobian of Lagrangian for each sample
dL = vmap(jacobian(lagrangian))(Y)

# Data: [x1, x2, x3, lambda1, lambda2, dL/dx1, dL/dx2, dL/dx3, dL/dlambda1, dL/dlambda2]
data = np.hstack([np.array(Y), np.array(dL)])
print(f"Data shape: {data.shape}")
print(f"Columns: [x1, x2, x3, λ1, λ2, ∂L/∂x1, ∂L/∂x2, ∂L/∂x3, ∂L/∂λ1, ∂L/∂λ2]")

In [ ]:
# Build GMM
gmm, info = best_gmm(data, verbose=True)
print(f"\nBest model: {info['best_k']} components")

In [ ]:
# Validate the model - predict derivatives given variables
pred = gmm.predict([0, 1, 2, 3, 4], data[:, 0:5])  # condition on x1,x2,x3,λ1,λ2

fig, axes = plt.subplots(1, 5, figsize=(16, 3))
labels = ['∂L/∂x1', '∂L/∂x2', '∂L/∂x3', '∂L/∂λ1', '∂L/∂λ2']

for i, (ax, label) in enumerate(zip(axes, labels)):
    true = data[:, 5+i]
    predicted = pred[:, i]
    ax.scatter(true, predicted, alpha=0.5, s=10)
    lims = [min(true.min(), predicted.min()), max(true.max(), predicted.max())]
    ax.plot(lims, lims, 'k--')
    ax.set_xlabel(f'True {label}')
    ax.set_ylabel(f'Predicted {label}')

plt.tight_layout()
plt.show()

In [ ]:
# Solve: condition on all 5 Lagrangian derivatives = 0
P = gmm.predict([5, 6, 7, 8, 9], np.array([[0.0, 0.0, 0.0, 0.0, 0.0]]))

x1_opt, x2_opt, x3_opt = P[0, 0:3]
lambda1_opt, lambda2_opt = P[0, 3], P[0, 4]

print("Generative solution (conditioning on ∇L = 0):")
print(f"  Reformate (x1) = {x1_opt:.4f}")
print(f"  Isomerate (x2) = {x2_opt:.4f}")
print(f"  Alkylate  (x3) = {x3_opt:.4f}")
print(f"  λ1 (octane)    = {lambda1_opt:.6f}")
print(f"  λ2 (mass bal)  = {lambda2_opt:.6f}")

# Verify constraints and cost
print(f"\nVerification:")
print(f"  Sum of fractions: {x1_opt + x2_opt + x3_opt:.4f} (should be 1.0)")
print(f"  RON achieved: {95*x1_opt + 82*x2_opt + 94*x3_opt:.2f} (target: 87)")
print(f"  Blend cost: ${2.50*x1_opt + 2.00*x2_opt + 3.00*x3_opt:.4f}/gal")

print(f"\nComparison with scipy: x = [{sol.x[0]:.4f}, {sol.x[1]:.4f}, {sol.x[2]:.4f}]")